In [105]:
!pip install scipy scikit-learn seaborn -q

In [106]:
import os
import tarfile
import urllib.request
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import RandAugment
from PIL import Image
from sklearn.metrics import confusion_matrix, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
images_url = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz"
labels_url = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat"
setid_url = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/setid.mat"

urllib.request.urlretrieve(images_url, "102flowers.tgz")
urllib.request.urlretrieve(labels_url, "imagelabels.mat")
urllib.request.urlretrieve(setid_url, "setid.mat")

with tarfile.open("102flowers.tgz", "r:gz") as tar:
    tar.extractall()

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6,1.0)),
    transforms.RandomHorizontalFlip(),
    RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
labels = scipy.io.loadmat("imagelabels.mat")["labels"][0]
setid = scipy.io.loadmat("setid.mat")

train_ids = setid["trnid"][0]
val_ids = setid["valid"][0]
test_ids = setid["tstid"][0]

class OxfordFlowers(Dataset):
    def __init__(self, ids, transform, all_labels):
        self.ids = ids
        self.transform = transform
        self.all_labels = all_labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_path = f"jpg/image_{img_id:05d}.jpg"
        image = Image.open(img_path).convert("RGB")
        label = self.all_labels[img_id-1] - 1
        return self.transform(image), label

train_dataset = OxfordFlowers(train_ids, train_transform, labels)
val_dataset = OxfordFlowers(val_ids, val_transform, labels)
test_dataset = OxfordFlowers(test_ids, val_transform, labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=384):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size,
                              stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1,2)
        return x

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim=384, heads=6, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim*mlp_ratio),
            nn.GELU(),
            nn.Linear(dim*mlp_ratio, dim)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, num_classes=102, depth=8):
        super().__init__()
        self.patch_embed = PatchEmbedding()
        num_patches = (224//16)**2

        self.cls_token = nn.Parameter(torch.zeros(1,1,384))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches+1, 384))

        self.blocks = nn.Sequential(*[
            TransformerBlock() for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(384)
        self.head = nn.Linear(384, num_classes)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)

        cls = self.cls_token.expand(B,-1,-1)
        x = torch.cat((cls,x), dim=1)
        x = x + self.pos_embed

        x = self.blocks(x)
        x = self.norm(x)

        return self.head(x[:,0])

In [ ]:
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam*x + (1-lam)*x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam*criterion(pred,y_a)+(1-lam)*criterion(pred,y_b)

In [ ]:
def accuracy(output, target, topk=(1,5)):
    maxk = max(topk)
    _, pred = output.topk(maxk,1,True,True)
    pred = pred.t()
    correct = pred.eq(target.view(1,-1).expand_as(pred))
    return [(correct[:k].reshape(-1).float().sum(0)*100/target.size(0)) for k in topk]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionTransformer().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

def adjust_lr(epoch, warmup=5):
    if epoch < warmup:
        lr = 3e-4*(epoch+1)/warmup
        for g in optimizer.param_groups:
            g['lr'] = lr
    else:
        scheduler.step()

In [ ]:
epochs = 100

for epoch in range(epochs):
    model.train()
    adjust_lr(epoch)

    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        imgs, y_a, y_b, lam = mixup_data(imgs, labels)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = mixup_loss(criterion, outputs, y_a, y_b, lam)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader)}")

In [ ]:
sum(p.numel() for p in model.parameters())

In [ ]:
model.eval()
top1_total = 0
top5_total = 0
count = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        acc1, acc5 = accuracy(outputs, labels)

        top1_total += acc1.item()
        top5_total += acc5.item()
        count += 1

print("Top-1:", top1_total/count)
print("Top-5:", top5_total/count)

In [ ]:
all_probs = []
all_targets = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = F.softmax(outputs, dim=1)

        all_probs.append(probs.cpu())
        all_targets.append(labels)

all_probs = torch.cat(all_probs).numpy()
all_targets = torch.cat(all_targets).numpy()

map_score = average_precision_score(all_targets, all_probs, average="macro")
print("mAP:", map_score)

In [ ]:
from sklearn.metrics import classification_report

all_preds = []
all_targets = []

classification = classification_report(all_targets, all_preds, output_dict=True)
print(classification)

In [ ]:
all_preds = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())

cm = confusion_matrix(all_targets, all_preds)
plt.tight_layout()
plt.figure(figsize=(10,8))
sns.heatmap(cm, cmap="Blues",annot=True)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
activations = None
gradients = None

def forward_hook(m,i,o):
    global activations
    activations = o

def backward_hook(m,gi,go):
    global gradients
    gradients = go[0]

model.patch_embed.proj.register_forward_hook(forward_hook)
model.patch_embed.proj.register_backward_hook(backward_hook)

In [ ]:
def gradcam(image, class_idx):
    model.eval()
    output = model(image)
    model.zero_grad()
    output[:,class_idx].backward()

    grads = gradients.mean(dim=[2,3], keepdim=True)
    cam = (grads*activations).sum(dim=1).squeeze()
    cam = torch.relu(cam)
    cam = cam-cam.min()
    cam = cam/cam.max()
    return cam.detach().cpu().numpy()

In [ ]:
from sklearn.metrics import average_precision_score
import torch.nn.functional as F

model.eval()
all_probs = []
all_targets = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)

        all_probs.append(probs.cpu())
        all_targets.append(labels)

all_probs = torch.cat(all_probs).numpy()
all_targets = torch.cat(all_targets).numpy()

map_score = average_precision_score(all_targets, all_probs, average="macro")
print("mAP:", map_score)

In [ ]:
model.eval()
correct = 0
total = 0
val_loss = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        val_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

val_acc = 100 * correct / total
val_loss /= len(val_loader)

print(f"Epoch {epoch+1}")
print(f"Train Loss: {total_loss/len(train_loader):.4f}")
print(f"Val Loss: {val_loss:.4f}")
print(f"Val Top-1 Accuracy: {val_acc:.2f}%")

In [ ]:
activations = None
gradients = None

def forward_hook(module, input, output):
    global activations
    activations = output

def backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0]


model.patch_embed.proj.register_forward_hook(forward_hook)
model.patch_embed.proj.register_backward_hook(backward_hook)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def generate_gradcam(model, image_tensor, class_idx=None):
    model.eval()

    output = model(image_tensor)

    if class_idx is None:
        class_idx = torch.argmax(output, dim=1).item()

    model.zero_grad()
    output[:, class_idx].backward()

    # activations shape: (B, C, H, W)
    grads = gradients.mean(dim=[2,3], keepdim=True)
    cam = (grads * activations).sum(dim=1).squeeze()

    cam = torch.relu(cam)
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

    cam = cam.detach().cpu().numpy()

    return cam, class_idx

In [ ]:
def show_gradcam(model, dataset, index=0):
    image, label = dataset[index]

    input_tensor = image.unsqueeze(0).to(device)

    cam, pred_class = generate_gradcam(model, input_tensor)

    # Resize CAM to image size
    cam_resized = cv2.resize(cam, (224,224))
    heatmap = cv2.applyColorMap(
        np.uint8(255 * cam_resized),
        cv2.COLORMAP_JET
    )

    heatmap = np.float32(heatmap) / 255

    image_np = image.permute(1,2,0).numpy()
    image_np = (image_np - image_np.min()) / (image_np.max() - image_np.min())

    overlay = heatmap * 0.4 + image_np
    overlay = overlay / overlay.max()

    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.title("Original")
    plt.imshow(image_np)
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.title("Grad-CAM")
    plt.imshow(cam_resized, cmap="jet")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.title(f"Overlay (Pred: {pred_class})")
    plt.imshow(overlay)
    plt.axis("off")

    plt.show()

In [ ]:

show_gradcam(model, test_dataset, index=5)

In [ ]:
for i in range(5):
    n = np.random.randint(0, len(test_dataset))
    show_gradcam(model, test_dataset, index=n)


In [ ]:
torch.save(model.state_dict(), "vit_flowers.pth")

In [ ]:
!pip install umap-learn -q